# Hierarchical Models

Partial pooling across groups — the "8 schools" style model. Group-level means share a common
prior whose parameters are themselves inferred from data.

**Model:**
$$\mu_{\text{global}} \sim \text{Normal}(0, 10)$$
$$\sigma_{\text{group}} \sim \text{HalfNormal}(5)$$
$$\mu_j \sim \text{Normal}(\mu_{\text{global}},\ \sigma_{\text{group}})$$
$$y_{ij} \sim \text{Normal}(\mu_j,\ \sigma_{\text{obs}})$$

In [ ]:
import numpy as np
import rustmc as rmc
import arviz as az
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

## Simulate Data

In [ ]:
np.random.seed(42)
J = 8
sigma_obs = 2.0
N_per_group = 30
mu_true = np.array([5.0, -1.0, 3.0, 0.0, 8.0, 2.0, -3.0, 6.0])

ys   = [np.random.normal(mu_true[j], sigma_obs, N_per_group) for j in range(J)]
data = {f"y_{j}": ys[j] for j in range(J)}

print(f"True mu_global:   {mu_true.mean():.2f}")
print(f"True sigma_group: {mu_true.std():.2f}")
print(f"True mu_j:        {mu_true.tolist()}")

## Build the Model

Hyperpriors are declared first. `mu_global` and `sigma_group` are `ParamRef` objects —
rustmc resolves them to graph nodes at sample time so gradients flow up to the hyperpriors.

In [ ]:
builder = rmc.ModelBuilder(data=data)

mu_global   = builder.normal_prior("mu_global",   mu=0.0, sigma=10.0)
sigma_group = builder.half_normal_prior("sigma_group", sigma=5.0)

mu_j = [
    builder.normal_prior(f"mu_{j}", mu=mu_global, sigma=sigma_group)
    for j in range(J)
]

for j in range(J):
    builder.normal_likelihood(f"obs_{j}", mu_expr=mu_j[j], sigma=sigma_obs, observed_key=f"y_{j}")

model = builder.build()

## Sample

In [ ]:
fit = rmc.sample(model_spec=model, chains=4, draws=2000, warmup=1000, seed=42)
print(fit.summary())

## ArviZ Diagnostics

In [ ]:
idata = fit.to_arviz()
az.summary(idata, round_to=4)

## Trace Plot

In [ ]:
# Show hyperparameters and a few group means
var_names = ["mu_global", "sigma_group", "mu_0", "mu_1", "mu_2"]
az.plot_trace(idata, var_names=var_names, figsize=(10, 8))
plt.tight_layout()
plt.show()

## Forest Plot — Partial Pooling

Shows all group means with HDI. The partial pooling effect (shrinkage toward the global mean)
is visible when comparing the pooled estimates to raw group sample means.

In [ ]:
group_vars = [f"mu_{j}" for j in range(J)]
az.plot_forest(idata, var_names=group_vars, combined=True, figsize=(8, 5))
plt.title("Group means — 94% HDI")
plt.axvline(fit.mean()["mu_global"], color="red", linestyle="--", alpha=0.6, label="mu_global")
plt.legend()
plt.tight_layout()
plt.show()

## Partial Pooling Effect

In [ ]:
means = fit.mean()
sample_means = [ys[j].mean() for j in range(J)]

print(f"Global mean estimate: {means['mu_global']:.2f}")
print(f"{'Group':<8} {'Raw':>8} {'Pooled':>10} {'True':>8}")
print("-" * 38)
for j in range(J):
    print(f"mu_{j}     {sample_means[j]:+8.2f} {means[f'mu_{j}']:+10.4f} {mu_true[j]:+8.2f}")

## Hyperparameter Posterior

In [ ]:
az.plot_posterior(idata, var_names=["mu_global", "sigma_group"], figsize=(10, 3))
plt.tight_layout()
plt.show()

print(f"mu_global:   true={mu_true.mean():.2f}  estimated={means['mu_global']:.4f} ± {fit.std()['mu_global']:.4f}")
print(f"sigma_group: true={mu_true.std():.2f}  estimated={means['sigma_group']:.4f} ± {fit.std()['sigma_group']:.4f}")